# Worflow of this notebook:

The required input for this workbook are `tif` files, each containing only 1 channel.

1) resave images as `tif`, splitting individual channels

2) segment image using a default or costum trained cellpose classifyer

3) delete `tif`s to save space


## Input
- `nd2` (spinning disk) or `h5` (STED) files -> will be resaved as `tif`, each conatining only 1 channel in step 1); alternatively you can provide such `tif`
- a cellpose classifyer (default or costum trained)

## Outpout
for each provided `tif`:
- segmentation mask as `png`
- segmentation mask as `npy`
- segmentation outline as `txt`
- vis folder with detection visualization for all `tif`s as `png` (good for checking segmentation results)

# 0) Imports and functions

The functions required for this to work are collected in the `pipelines/fish_utils` folder. Download the folder from `/../` in this repository and `sys.path.append(/path/to/fish_utils/)`. You can skip this if you are providing `tif`s.

In [ ]:
import h5py as h5
import json
import os
from glob import glob
import sys
import tifffile
import pandas as pd
import numpy as np
import torch

sys.path.append('../pipelines/')
from fish_utils.resave import resave_h5, remove_tifs

from natsort import natsorted
import matplotlib.pyplot as plt
from skimage.io import imread

from cellpose import models, io
from cellpose.io import imread
from cellpose import plot

# 1) Resave h5/nd2 as tif

In [ ]:
# folder containing the images
files = "/data/agl_data/.../imgs/"

# choose 
resave_h5(files)
# resave_nd2 (files)

# make sure to delete the tifs after you are done with the analysis!!!

# 2) Segmentation

In [ ]:
# to do the segmentaion fast work on the gpu
device = torch.device('cuda:0')
torch.cuda.is_available()

In [ ]:
# model = models.Cellpose(model_type='cyto', device=device)
model = models.CellposeModel(model_type = "microfluidic_20230828", device=device)

files = glob("/data/agl_data/../tif/*_ch0.tif")
out_folder = "/data/agl_data/../segmentation/vis/"

chan = [[0,0]]
diams = 115

# or in a loop
for filename in files:
    
    out = filename.replace("/tif", "/segmentation")
    create_folder(out_folder)
    img = io.imread(filename).max(axis=0)
    
    masks, flows, styles= model.eval(img, channels=chan,flow_threshold=0.5)

    # save results so you can load in gui
    io.masks_flows_to_seg(img, masks, flows, diams, out)

    # save results as png
    io.save_to_png(img, masks, flows, out)
    
    # plot segmentation o check
    fig = plt.figure(figsize=(12,3.5))
    plot.show_segmentation(fig, img, masks, flows[0], channels=chan)
    plt.tight_layout()
    fig.savefig(f"{out_folder}/{os.path.basename(out)}.png",dpi=300)
    plt.close(fig)

# 3) Remove tif folder
Please always remove tifs after you are done with the analysis to save space.

In [ ]:
folder = "/data/agl_data/.../tif/"

remove_tifs(folder)